In [ ]:
import pandas as pd
import numpy as np
import json

In [ ]:
#Parameters
raw_csv_path = '/content/RAW_Results.csv'
age_csv_path = '/content/Age.csv'
config_json_path = '/content/canmovesRunSpecConfig.json'
fuel_distribution_csv_path = '/content/Fuel Distribution_Basic.csv'
output_csv_path_model = '/content/df_emission_model.csv'
output_csv_path_source = '/content/df_emission_source.csv'
output_csv_path_fuel = '/content/df_emission_fuel.csv'
output_csv_path_total = '/content/df_emission_total.csv'
output_csv_path_total_chart = '/content/df_emission_chart_total.csv'
output_csv_path_fuel_chart = '/content/df_emission_chart_by_fuel.csv'
output_csv_path_model_chart = '/content/df_emission_chart_by_model.csv'
output_csv_path_source_chart = '/content/df_emission_chart_by_source.csv'

#Functions

In [ ]:
def get_pollutant_ids():
    return [
        "Energy (kWh)",
        "Gasoline (kg)",
        "Diesel (kg)",
        "NG (kg)",
        "LPG (kg)",
        "Methanol (kg)",
        "Ethanol (kg)",
        "Electricity (kWh)",
        "Elec. Plant CO2 (kg)",
        "Elec. Plant NOx (kg)",
        "Elec. Plant SO2 (kg)",
        "Tailpipe CO2 (kg)",
        "CO (kg)",
        "Tailpipe NMHC (kg)",
        "Tailpipe NOx (kg)",
        "Tailpipe Particulate Matter (g)",
        "Sub 2.5 um Tailpipe PM (g)",
        "Tailpipe SO2 (kg)",
        "NH3 (kg)",
        "Brake and Tire Particulate Matter (kg)",
    ]

In [ ]:
def get_vehicle_types():
    return [
        ("LDV-mini", "LDV-Mini", "LDV-mini"),
        ("LDV-Eco", "LDV-Economy", "LDV-Eco"),
        ("LDV-Large", "LDV-Large", "LDV-Large"),
        ("LDT1", "LDT1", "LDT1"),
        ("LDT2", "LDT2", "LDT2"),
        ("LDT3", "LDT3", "LDT3"),
        ("LDT4", "LDT4", "LDT4"),
        ("HDV2b3", "HDV2b", "HDV2b3"),
        ("HDV3", "HDV3", "HDV3"),
        ("HDV4", "HDV4", "HDV4"),
        ("HDV5", "HDV5", "HDV5"),
        ("HDV6", "HDV6", "HDV6"),
        ("HDV7", "HDV7", "HDV7"),
        ("HDV8a", "HDV8a", "HDV8a"),
        ("HDV8b", "HDV8b", "HDV8b"),
        ("Bus - SS", "Bus - SS", "Short School"),
        ("Bus - SL", "Bus - SL", "Blong Schoolus"),
        ("Bus - TO", "Bus - TO", "Old Transit"),
        ("Bus - TN", "Bus - TN", "New Tranist"),
        ("Bus - TL", "Bus - TL", "Long Transit"),
        ("Bus - TS", "Bus - TS", "Short Transit"),
    ]

In [ ]:
def clean_header(name):
    name = str(name).strip()
    name = " ".join(name.split())
    name = name.replace("Tailpipe Nox", "Tailpipe NOx")
    return name

In [ ]:
def read_selected_year(config_json_path):
    with open(config_json_path, "r") as file:
        config = json.load(file)

    return int(config["domainInfo"]["selectedYear"])

In [ ]:
def read_link_type_mapping(config_json_path):
    with open(config_json_path, "r") as file:
        config = json.load(file)

    link_type_mapping = {}

    for index, category in enumerate(config["linkCategories"], start=1):
        if category["enabled"]:
            link_type_mapping[index] = category["name"]

    return link_type_mapping

In [ ]:
def prepare_raw_results(raw_csv_path, pollutant_ids=None):
    if pollutant_ids is None:
        pollutant_ids = get_pollutant_ids()

    raw_df = pd.read_csv(raw_csv_path)
    raw_df = raw_df.dropna()
    raw_df.columns = [clean_header(col) for col in raw_df.columns]

    raw_df["inode"] = raw_df["inode"].astype(int)
    raw_df["jnode"] = raw_df["jnode"].astype(int)

    raw_df["link"] = raw_df["inode"].astype(str) + "-" + raw_df["jnode"].astype(str)

    vehicle_types = get_vehicle_types()
    renamed_columns = {}

    for col in raw_df.columns:
        for vehicle_type, raw_prefix, age_header in vehicle_types:
            raw_prefix = clean_header(raw_prefix)

            if col.startswith(raw_prefix + " "):
                pollutant_id = col.replace(raw_prefix + " ", "", 1)

                if pollutant_id in pollutant_ids:
                    renamed_columns[col] = f"{vehicle_type} {pollutant_id}"

                break

    standardized_df = raw_df[["link"] + list(renamed_columns.keys())].rename(columns=renamed_columns)

    long_df = standardized_df.melt(
        id_vars="link",
        var_name="vehicle_pollutant",
        value_name="original_value",
    )

    vehicle_pollutant_pairs = []

    for col in long_df["vehicle_pollutant"]:
        for pollutant_id in pollutant_ids:
            suffix = " " + pollutant_id

            if col.endswith(suffix):
                vehicle_type = col[:-len(suffix)]
                vehicle_pollutant_pairs.append((vehicle_type, pollutant_id))
                break

    long_df[["vehicle_type", "pollutantID"]] = pd.DataFrame(vehicle_pollutant_pairs, index=long_df.index)

    long_df["original_value"] = pd.to_numeric(long_df["original_value"], errors="coerce")

    kwh_mask = long_df["pollutantID"].str.endswith("(kWh)")
    kg_mask = long_df["pollutantID"].str.endswith("(kg)")

    long_df.loc[kwh_mask, "original_value"] = long_df.loc[kwh_mask, "original_value"] * 1000
    long_df.loc[kg_mask, "original_value"] = long_df.loc[kg_mask, "original_value"] * 1000

    long_df["pollutantID"] = long_df["pollutantID"].replace(
        {
            "Energy (kWh)": "Energy (Wh)",
            "Electricity (kWh)": "Electricity (Wh)",
            "Gasoline (kg)": "Gasoline (g)",
            "Diesel (kg)": "Diesel (g)",
            "NG (kg)": "NG (g)",
            "LPG (kg)": "LPG (g)",
            "Methanol (kg)": "Methanol (g)",
            "Ethanol (kg)": "Ethanol (g)",
            "Elec. Plant CO2 (kg)": "Elec. Plant CO2 (g)",
            "Elec. Plant NOx (kg)": "Elec. Plant NOx (g)",
            "Elec. Plant SO2 (kg)": "Elec. Plant SO2 (g)",
            "Tailpipe CO2 (kg)": "Tailpipe CO2 (g)",
            "CO (kg)": "Tailpipe CO (g)",
            "Tailpipe NMHC (kg)": "Tailpipe NMHC (g)",
            "Tailpipe NOx (kg)": "Tailpipe NOx (g)",
            "Tailpipe SO2 (kg)": "Tailpipe SO2 (g)",
            "NH3 (kg)": "Tailpipe NH3 (g)",
            "Brake and Tire Particulate Matter (kg)": "Brake and Tire PM (g)",
            "Sub 2.5 um Tailpipe PM (g)": "Tailpipe PM2.5 (g)",
            "Tailpipe Particulate Matter (g)": "Tailpipe PM (g)",
        }
    )

    return long_df[["link", "vehicle_type", "pollutantID", "original_value"]]

In [ ]:
def summarize_raw_results_by_link_type(raw_csv_path, config_json_path, pollutant_ids=None):

    raw_long_df = prepare_raw_results(raw_csv_path=raw_csv_path, pollutant_ids=pollutant_ids)

    raw_df = pd.read_csv(raw_csv_path)
    raw_df = raw_df.dropna()
    raw_df.columns = [clean_header(col) for col in raw_df.columns]

    raw_df["inode"] = raw_df["inode"].astype(int)
    raw_df["jnode"] = raw_df["jnode"].astype(int)

    raw_df["link"] = raw_df["inode"].astype(str) + "-" + raw_df["jnode"].astype(str)

    link_type_mapping = read_link_type_mapping(config_json_path)

    raw_df["Link Type"] = raw_df["Link Type"].astype(int).map(link_type_mapping)

    link_type_df = raw_df[["link", "Link Type"]].drop_duplicates()

    summary_df = raw_long_df.merge(link_type_df, on="link", how="left")

    summary_df = (summary_df.groupby(["link", "Link Type", "pollutantID"], as_index=False)["original_value"].sum())

    summary_df = summary_df.rename(columns={"link": "linkID", "Link Type": "linkTypeID", "original_value": "emissionQuant"})

    return summary_df[["linkID", "linkTypeID", "pollutantID", "emissionQuant"]]

In [ ]:
def prepare_age_distribution(age_csv_path):
    age_df = pd.read_csv(age_csv_path)
    age_df.columns = [clean_header(col) for col in age_df.columns]

    vehicle_types = get_vehicle_types()
    age_rename = {clean_header(age_header): vehicle_type for vehicle_type, raw_prefix, age_header in vehicle_types}

    age_df = age_df.rename(columns=age_rename)
    age_df = age_df.rename(columns={"Vehicle Age": "vehicle_age"})

    vehicle_columns = [vehicle_type for vehicle_type, raw_prefix, age_header in vehicle_types]
    age_df = age_df[["vehicle_age"] + vehicle_columns]

    long_df = age_df.melt(id_vars="vehicle_age", value_vars=vehicle_columns, var_name="vehicle_type", value_name="age_fraction")

    long_df["vehicle_age"] = pd.to_numeric(long_df["vehicle_age"], errors="coerce")
    long_df["age_fraction"] = pd.to_numeric(long_df["age_fraction"], errors="coerce")

    return long_df[["vehicle_type", "vehicle_age", "age_fraction"]]

In [ ]:
def expand_using_age_distribution(raw_long_df, age_long_df):
    expanded_df = raw_long_df.merge(age_long_df, on="vehicle_type", how="left")
    expanded_df["expanded_value"] = expanded_df["original_value"] * expanded_df["age_fraction"]

    return expanded_df[["link", "vehicle_type", "vehicle_age", "pollutantID", "expanded_value"]]

In [ ]:
def summarize_expanded_results_model(expanded_df, config_json_path):
    summary_ready_df = expanded_df.copy()

    selected_year = read_selected_year(config_json_path)

    summary_ready_df["modelYearID"] = selected_year - summary_ready_df["vehicle_age"]
    summary_ready_df["modelYearID"] = summary_ready_df["modelYearID"].astype(int)

    summary_ready_df = summary_ready_df.rename(
        columns={
            "link": "linkID",
            "expanded_value": "emissionQuant",
        }
    )

    summarized_df = (summary_ready_df.groupby(["linkID", "modelYearID", "pollutantID"], as_index=False)["emissionQuant"].sum())

    return summarized_df[["linkID", "modelYearID", "pollutantID", "emissionQuant"]]

In [ ]:
def summarize_expanded_results_source(expanded_df, config_json_path):
    summary_ready_df = expanded_df.copy()

    summary_ready_df = summary_ready_df.rename(
        columns={
            "link": "linkID",
            "expanded_value": "emissionQuant",
            "vehicle_type": "sourceTypeID",
        }
    )

    summarized_df = (summary_ready_df.groupby(["linkID", "sourceTypeID", "pollutantID"], as_index=False)["emissionQuant"].sum())

    return summarized_df[["linkID", "sourceTypeID", "pollutantID", "emissionQuant"]]

In [ ]:
def prepare_fuel_distribution(fuel_distribution_csv_path):
    fuel_df = pd.read_csv(fuel_distribution_csv_path)
    fuel_df.columns = [clean_header(col) for col in fuel_df.columns]

    fuel_columns = {
        "Percent Gasoline": "Gasoline",
        "Percent Diesel": "Diesel",
        "Percent Nat. Gas": "Nat. Gas",
        "Percent Propane": "Propane",
        "Percent Methanol": "Methanol",
        "percent Ethanol": "Ethanol",
    }

    fuel_column_names = list(fuel_columns.keys())

    fuel_df = fuel_df[
        ["Subclass", "Class #", "Sub-class #"] + fuel_column_names
    ].copy()

    fuel_sum = fuel_df[fuel_column_names].sum(axis=1)

    zero_distribution_rows = fuel_sum == 0

    for index, row in fuel_df[zero_distribution_rows].iterrows():
        class_number = row["Class #"]

        base_distribution = fuel_df[
            (fuel_df["Class #"] == class_number) &
            (fuel_df["Sub-class #"] == 1)
        ]

        if not base_distribution.empty:
            fuel_df.loc[index, fuel_column_names] = base_distribution.iloc[0][fuel_column_names].values

    fuel_sum = fuel_df[fuel_column_names].sum(axis=1)

    fuel_df[fuel_column_names] = (
        fuel_df[fuel_column_names]
        .div(fuel_sum.replace(0, np.nan), axis=0)
        .fillna(0)
    )

    fuel_df["Subclass"] = fuel_df["Subclass"].replace(
        {
            "LDV-Econ": "LDV-Eco",
            "HDV2b": "HDV2b3",
            "Short School": "Bus - SS",
            "Long School": "Bus - SL",
            "Old Transit": "Bus - TO",
            "New Transit": "Bus - TN",
            "Long transit": "Bus - TL",
            "Short Transit": "Bus - TS",
        }
    )

    fuel_long_df = fuel_df.melt(
        id_vars="Subclass",
        value_vars=fuel_column_names,
        var_name="fuel_column",
        value_name="fuel_fraction",
    )

    fuel_long_df["fuel_type"] = fuel_long_df["fuel_column"].replace(fuel_columns)

    fuel_long_df = fuel_long_df.rename(columns={"Subclass": "vehicle_type"})

    return fuel_long_df[["vehicle_type", "fuel_type", "fuel_fraction"]]

In [ ]:
def expand_tailpipe_by_fuel_distribution(raw_long_df, fuel_distribution_df):
    tailpipe_df = raw_long_df[raw_long_df["pollutantID"].str.contains("Tailpipe", case=False, na=False)].copy()

    expanded_fuel_df = tailpipe_df.merge(fuel_distribution_df, on="vehicle_type", how="left")

    expanded_fuel_df["fuel_expanded_value"] = (expanded_fuel_df["original_value"] * expanded_fuel_df["fuel_fraction"])

    return expanded_fuel_df[["link", "vehicle_type", "fuel_type", "pollutantID", "fuel_fraction", "fuel_expanded_value"]]

In [ ]:
def summarize_tailpipe_fuel_by_vehicle_type(tailpipe_fuel_expanded_df):
    summarized_df = tailpipe_fuel_expanded_df.groupby(["link", "fuel_type", "pollutantID"], as_index=False)["fuel_expanded_value"].sum()

    summarized_df = summarized_df.rename(columns={"link": "linkID", "fuel_type": "fuelTypeID", "fuel_expanded_value": "emissionQuant"})

    return summarized_df[["linkID", "fuelTypeID", "pollutantID", "emissionQuant"]]

In [ ]:
def reshape_link_type_summary_total(link_type_summary_df):
    reshaped_df = link_type_summary_df.pivot_table(
        index=["linkTypeID"],
        columns="pollutantID",
        values="emissionQuant",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()

    reshaped_df.columns.name = None

    return reshaped_df

In [ ]:
def reshape_link_type_summary_fuel(df_emission_fuel):
    reshaped_df = df_emission_fuel.pivot_table(
        index=["fuelTypeID"],
        columns="pollutantID",
        values="emissionQuant",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()

    reshaped_df.columns.name = None

    return reshaped_df

In [ ]:
def reshape_link_type_summary_model(df_emission_model):
    reshaped_df = df_emission_model.pivot_table(
        index=["modelYearID"],
        columns="pollutantID",
        values="emissionQuant",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()

    reshaped_df.columns.name = None

    return reshaped_df

In [ ]:
def reshape_link_type_summary_source(df_emission_model):
    reshaped_df = df_emission_model.pivot_table(
        index=["sourceTypeID"],
        columns="pollutantID",
        values="emissionQuant",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()

    reshaped_df.columns.name = None

    return reshaped_df

#Main_Code

In [ ]:
if __name__ == "__main__":

    pollutant_ids = get_pollutant_ids()

    raw_long_df = prepare_raw_results(raw_csv_path, pollutant_ids)

    age_long_df = prepare_age_distribution(age_csv_path)

    expanded_df = expand_using_age_distribution(raw_long_df, age_long_df)

    df_emission_model = summarize_expanded_results_model(expanded_df, config_json_path)

    df_emission_source = summarize_expanded_results_source(expanded_df, config_json_path)

    fuel_distribution_df = prepare_fuel_distribution(fuel_distribution_csv_path)

    tailpipe_fuel_expanded_df = expand_tailpipe_by_fuel_distribution(raw_long_df,fuel_distribution_df)

    df_emission_fuel = summarize_tailpipe_fuel_by_vehicle_type(tailpipe_fuel_expanded_df)

    df_emission_total = summarize_raw_results_by_link_type(raw_csv_path, config_json_path)

    df_emission_chart_total = reshape_link_type_summary_total(df_emission_total)

    df_emission_chart_by_fuel = reshape_link_type_summary_fuel(df_emission_fuel)

    df_emission_chart_by_model = reshape_link_type_summary_model(df_emission_model)

    df_emission_chart_by_source = reshape_link_type_summary_source(df_emission_source)

#Print

In [ ]:
df_emission_model.to_csv(output_csv_path_model, index=False)
df_emission_source.to_csv(output_csv_path_source, index=False)
df_emission_fuel.to_csv(output_csv_path_fuel, index=False)
df_emission_total.to_csv(output_csv_path_total, index=False)
df_emission_chart_total.to_csv(output_csv_path_total_chart, index=False)
df_emission_chart_by_fuel.to_csv(output_csv_path_fuel_chart, index=False)
df_emission_chart_by_model.to_csv(output_csv_path_model_chart, index=False)
df_emission_chart_by_source.to_csv(output_csv_path_source_chart, index=False)